# Supply Chain Operations Dashboard — Data Cleaning & Preparation

**Tools:** Python · pandas · NumPy · re  
**Dataset:** DataCo Smart Supply Chain (180,000+ orders, 2015–2018)  
**Project:** Part 1 of 3 — Cleaning Notebook → EDA Notebook → Power BI Dashboard

## Background

This project is part of a data analyst portfolio and is a full-stack analytical workflow from raw data to a deployed Power BI dashboard, covering the operational questions that matter most in supply chain — delivery performance, late order risk, revenue concentration, and regional logistics patterns.

The dataset used here is the DataCo Smart Supply Chain dataset, a publicly available e-commerce supply chain dataset covering 180,000+ order lines across four years, multiple global markets, and several product departments. It contains customer information, product details, order financials, and shipping logistics in a single denormalized flat file.

## Why Clean in Python Before Loading to Power BI

A common shortcut is to load raw data directly into Power BI and handle cleaning inside Power Query. That works for simple datasets. It is the wrong approach here for three reasons.

First, the raw file is a single denormalized table with 52 columns where customer, product, and shipping information is repeated on every row. Loading this directly into Power BI produces a bloated, unstructured model with no relationships. The correct approach is to split this flat file into a fact table and dimension tables — a star schema — before it ever touches Power BI.

Second, the datetime columns are stored as strings. Power Query can parse dates, but engineering derived features — actual lead time, delivery delay in days, performance bands — is faster, more transparent, and more reproducible in Python.

## What This Notebook Does

This notebook works through the following steps in order:

1. **Load & Inspect** — understand the raw file structure, column types, null counts, and duplicates before touching anything
2. **Datetime Parsing & Engineering** — convert date strings, calculate actual lead times, delivery delay, and bin performance into operational bands
3. **String & Regex Cleaning** — standardise inconsistent text fields; parse URL structure from the access log file as a regex demonstration
4. **Feature Engineering** — derive profit margin, on-time delivery flag, and order value bands from existing columns
5. **Null Handling & Validation** — document every null treatment decision explicitly
6. **Star Schema Split** — decompose the flat file into `fact_orders`, `dim_customer`, `dim_product`, and `dim_shipping` and export as clean CSVs

The output of this notebook is four clean CSVs saved to `Data/Cleaned/`. Those files are what Power BI loads — the raw file is never modified.

## Section 1 — Load & Inspect

Before any cleaning begins, the goal is to understand exactly what the raw data contains. This means checking the shape, column data types, null counts, and whether any duplicate rows exist. No modifications are made in this section — only observation and documentation of what needs to be addressed in subsequent steps.

The file requires `latin-1` encoding because it contains characters outside standard UTF-8 — a common issue with datasets exported from older ERP or e-commerce systems.

In [1]:
import pandas as pd
import numpy as np
import re

# Load raw file — latin-1 encoding required due to special characters in product names
df = pd.read_csv('Data/DataCoSupplyChainDataset.csv', encoding='latin-1')

# Shape
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# Column names and data types
print("=== Column Data Types ===")
print(df.dtypes.to_string())

# Null counts — only show columns with at least one null
print("\n=== Null Counts (columns with nulls only) ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())

# Duplicate rows
dupes = df.duplicated().sum()
print(f"\n=== Duplicate Rows: {dupes} ===")

# First 5 rows
print("\n=== First 5 Rows ===")
df.head()

Shape: 180,519 rows × 53 columns

=== Column Data Types ===
Type                                 str
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                      str
Late_delivery_risk                 int64
Category Id                        int64
Category Name                        str
Customer City                        str
Customer Country                     str
Customer Email                       str
Customer Fname                       str
Customer Id                        int64
Customer Lname                       str
Customer Password                    str
Customer Segment                     str
Customer State                       str
Customer Street                      str
Customer Zipcode                 float64
Department Id                      int64
Department Name                      str
Latitude                         float

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


### What the Inspection Tells Us

The raw file contains 180,519 rows and 53 columns with no duplicate rows. Four columns have nulls and each requires a different treatment:

- **`Product Description`** — 180,519 nulls. The entire column is empty. It will be dropped.
- **`Order Zipcode`** — 155,679 nulls, which is 86% of rows. The column is not recoverable and will be dropped.
- **`Customer Lname`** — 8 nulls. Negligible. Rows will be dropped.
- **`Customer Zipcode`** — 3 nulls. Negligible. Rows will be dropped.

Three additional columns will be dropped for reasons unrelated to nulls:
- **`Customer Password`** — contains real-looking password data. No analytical value and should not exist in any portfolio dataset.
- **`Customer Email`** — same reasoning. PII with no analytical use.
- **`Product Image`** — image URLs. No analytical value.

The two date columns — `order date (DateOrders)` and `shipping date (DateOrders)` — are stored as strings in mixed datetime format (`M/D/YYYY HH:MM`). These will be parsed in Section 2.

## Section 2 — Datetime Parsing & Engineering

The two date columns arrive as strings in `M/D/YYYY HH:MM` format. This section parses them into proper datetime objects and then engineers derived columns that are central to the operational analysis.

Before applying any binning or categorisation, we first explore the actual distribution of the data. Bin boundaries should be justified by what the data contains — not set arbitrarily in advance.

In [2]:
# Drop columns before datetime work — cleaner to remove noise first
cols_to_drop = [
    'Product Description',  # 100% null
    'Order Zipcode',        # 86% null — unrecoverable
    'Customer Password',    # PII — no analytical value
    'Customer Email',       # PII — no analytical value
    'Product Image'         # URL strings — no analytical value
]
df.drop(columns=cols_to_drop, inplace=True)

# Drop rows with negligible nulls in Customer Lname and Customer Zipcode
df.dropna(subset=['Customer Lname', 'Customer Zipcode'], inplace=True)
print(f"Shape after drops: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Parse date columns from string to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'], format='mixed'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'], format='mixed'
)

print(f"\nOrder date range:    {df['order date (DateOrders)'].min().date()} → {df['order date (DateOrders)'].max().date()}")
print(f"Shipping date range: {df['shipping date (DateOrders)'].min().date()} → {df['shipping date (DateOrders)'].max().date()}")

Shape after drops: 180,508 rows × 48 columns

Order date range:    2015-01-01 → 2018-01-31
Shipping date range: 2015-01-03 → 2018-02-06


### Explore Lead Time Distribution Before Binning

We calculate actual lead time first — the number of days between order placement and shipment — then examine its distribution before deciding how to bin it. This ensures our performance bands reflect how this specific dataset is distributed rather than a generic assumption.

In [3]:
# Engineer actual lead time in days
df['actual_lead_time'] = (
    df['shipping date (DateOrders)'] - df['order date (DateOrders)']
).dt.days

print("=== Actual Lead Time Distribution ===")
print(df['actual_lead_time'].describe())

print("\n=== Value Counts ===")
print(df['actual_lead_time'].value_counts().sort_index().to_string())

=== Actual Lead Time Distribution ===
count    180508.000000
mean          3.471874
std           1.670495
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: actual_lead_time, dtype: float64

=== Value Counts ===
actual_lead_time
0     9737
2    56613
3    28764
4    28510
5    28161
6    28723


### Observation & Binning Decision

The lead time range is 0 to 6 days. There are no 1-day orders. The median is 3 days, meaning half of all orders ship within 3 days of being placed.

Given this distribution, we define four performance bands with boundaries set at the natural breaks in the data:

| Band | Days | Reasoning |
|---|---|---|
| Same Day | 0 | Orders dispatched on the day they are placed |
| Fast | 1–2 | Below the 25th percentile — top quartile performance |
| Standard | 3–4 | Centred on the median — typical operational throughput |
| Slow | 5–6 | Above the 75th percentile — underperforming orders |

All four bands are populated. This categorisation gives operations managers a meaningful slicer in Power BI — "how many orders shipped fast this quarter versus slow?" — rather than a continuous number that requires interpretation.

In [4]:
# Delivery delay: positive = late, negative = early, zero = on time
df['delivery_delay_days'] = (
    df['actual_lead_time'] - df['Days for shipment (scheduled)']
)

# Recalculate late flag from our own delay column
df['is_late'] = (df['delivery_delay_days'] > 0).astype(int)

# Validate against dataset's own Late_delivery_risk column
agreement = (df['is_late'] == df['Late_delivery_risk']).mean() * 100
print(f"Late delivery flag agreement rate: {agreement:.1f}%")

# Lead time performance bands — boundaries justified by distribution above
df['lead_time_band'] = pd.cut(
    df['actual_lead_time'],
    bins=[-np.inf, 0, 2, 4, np.inf],
    labels=['Same Day', 'Fast', 'Standard', 'Slow']
)

print("\nLead time band distribution:")
print(df['lead_time_band'].value_counts().sort_index().to_string())

# Time components for Power BI trend analysis
df['order_year']      = df['order date (DateOrders)'].dt.year
df['order_month']     = df['order date (DateOrders)'].dt.month
df['order_quarter']   = df['order date (DateOrders)'].dt.to_period('Q').astype(str)
df['order_yearmonth'] = df['order date (DateOrders)'].dt.to_period('M').astype(str)

print("\nOrder year distribution:")
print(df['order_year'].value_counts().sort_index().to_string())

Late delivery flag agreement rate: 95.2%

Lead time band distribution:
lead_time_band
Same Day     9737
Fast        56613
Standard    57274
Slow        56884

Order year distribution:
order_year
2015    62650
2016    62550
2017    53186
2018     2122


### Section 2 Findings

**Date parsing** confirmed both date columns span January 2015 through early 2018. The shipping date extends slightly beyond the order date range, which is expected — orders placed in late January 2018 ship into February.

**Lead time distribution** shows all orders dispatch within 0 to 6 days with no 1-day orders present. The distribution is notably even across the upper three bands — Fast, Standard, and Slow each contain roughly 56,000–57,000 orders, approximately 31% each. Same Day orders account for 5.4% of all orders. This even spread means the lead time band column will produce balanced, readable visuals in Power BI rather than one dominant category.

**Delivery delay flag** agrees with the dataset's own `Late_delivery_risk` column 95.2% of the time. The 4.8% disagreement is expected — the dataset's flag appears to be pre-calculated at order creation using scheduled dates, while our `is_late` flag is calculated from actual shipping dates after the fact. Our calculated flag is the more analytically honest version and is what we will use going forward.

**Year distribution** reveals that 2018 contains only 2,122 orders — roughly 3 weeks of data covering January only. This is important context for Power BI: year-over-year comparisons must exclude 2018 from any annual totals, and the forecast built into the dashboard should be anchored on the 2015–2017 complete years only.

## Section 3 — String & Regex Cleaning

This section has two distinct parts with different purposes.

**Part A** cleans text fields in the main dataset that will be used as categorical dimensions in Power BI — city names, regions, segments, and delivery status labels. Inconsistent capitalisation and whitespace in these columns cause Power BI to treat identical values as separate categories, which breaks regional and segment-level analysis. This cleaning is analytically necessary.

**Part B** demonstrates regex pattern extraction on the access log URL column. The URLs follow a structured path format that encodes department, category, and product name. We extract these components using regex and decode the percent-encoding that replaces spaces and special characters in URLs. This output does not feed into the Power BI dashboard but demonstrates a practically relevant data extraction technique — the kind of parsing that appears regularly when working with web logs, API responses, or ERP export strings.

### Part A — Text Field Standardisation (Main Dataset)

Target columns and the specific problems each has:
- **`Customer City`** — mixed case, leading/trailing whitespace
- **`Order Region`** — inconsistent capitalisation, extra internal spaces
- **`Customer Segment`** — mixed case
- **`Delivery Status`** — mixed case, needs to be consistent for Power BI slicer labels
- **`Type`** — payment type field, already uppercase but needs whitespace check
- **`Shipping Mode`** — mixed case, extra whitespace possible

In [5]:
# Helper function to apply standard text cleaning
# strip whitespace → fix internal multiple spaces → title case
def clean_text(series):
    return (series
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .str.title())

# Apply to all target categorical columns
text_cols = [
    'Customer City',
    'Order Region',
    'Customer Segment',
    'Delivery Status',
    'Shipping Mode',
    'Department Name',
    'Category Name',
    'Order Status',
    'Market'
]

for col in text_cols:
    df[col] = clean_text(df[col])

# Type column — payment type, keep uppercase
df['Type'] = df['Type'].str.strip().str.upper()

# Verify — print unique values for key columns
print("=== Customer Segment unique values ===")
print(sorted(df['Customer Segment'].unique()))

print("\n=== Delivery Status unique values ===")
print(sorted(df['Delivery Status'].unique()))

print("\n=== Shipping Mode unique values ===")
print(sorted(df['Shipping Mode'].unique()))

print("\n=== Market unique values ===")
print(sorted(df['Market'].unique()))

print("\n=== Order Region unique values ===")
print(sorted(df['Order Region'].unique()))

=== Customer Segment unique values ===
['Consumer', 'Corporate', 'Home Office']

=== Delivery Status unique values ===
['Advance Shipping', 'Late Delivery', 'Shipping Canceled', 'Shipping On Time']

=== Shipping Mode unique values ===
['First Class', 'Same Day', 'Second Class', 'Standard Class']

=== Market unique values ===
['Africa', 'Europe', 'Latam', 'Pacific Asia', 'Usca']

=== Order Region unique values ===
['Canada', 'Caribbean', 'Central Africa', 'Central America', 'Central Asia', 'East Africa', 'East Of Usa', 'Eastern Asia', 'Eastern Europe', 'North Africa', 'Northern Europe', 'Oceania', 'South America', 'South Asia', 'South Of Usa', 'Southeast Asia', 'Southern Africa', 'Southern Europe', 'Us Center', 'West Africa', 'West Asia', 'West Of Usa', 'Western Europe']


In [6]:
# str.title() incorrectly lowercases acronyms — fix manually
# Market: USCA
df['Market'] = df['Market'].str.replace('Usca', 'USCA', regex=False)

# Order Region: USA and US references
df['Order Region'] = df['Order Region'].str.replace('Usa', 'USA', regex=False)
df['Order Region'] = df['Order Region'].str.replace('Us Center', 'US Center', regex=False)
df['Order Region'] = df['Order Region'].str.replace('East Of USA', 'East of USA', regex=False)
df['Order Region'] = df['Order Region'].str.replace('South Of USA', 'South of USA', regex=False)
df['Order Region'] = df['Order Region'].str.replace('West Of USA', 'West of USA', regex=False)

# Verify
print("=== Market unique values after fix ===")
print(sorted(df['Market'].unique()))

print("\n=== Order Region unique values after fix ===")
print(sorted(df['Order Region'].unique()))

=== Market unique values after fix ===
['Africa', 'Europe', 'Latam', 'Pacific Asia', 'USCA']

=== Order Region unique values after fix ===
['Canada', 'Caribbean', 'Central Africa', 'Central America', 'Central Asia', 'East Africa', 'East of USA', 'Eastern Asia', 'Eastern Europe', 'North Africa', 'Northern Europe', 'Oceania', 'South America', 'South Asia', 'South of USA', 'Southeast Asia', 'Southern Africa', 'Southern Europe', 'US Center', 'West Africa', 'West Asia', 'West of USA', 'Western Europe']


### Part B — Regex URL Parsing (Access Logs)

The access log file contains a URL column with entries following this structure:

`/department/fan%20shop/category/water%20sports/product/Pelican%20Sunstream%20100%20Kayak`

Each URL encodes three pieces of information — department, category, and product name — separated by path segments. The `%20` sequences are percent-encoded spaces, a standard URL encoding format.

We use regex to extract the three components from the path structure, then decode the percent-encoding to recover readable text. This is the same pattern you encounter when parsing API endpoint logs, web server access files, or any system that URL-encodes its routing parameters.

In [7]:
# Load access log file
logs = pd.read_csv('Data/tokenized_access_logs.csv', encoding='latin-1')

print(f"Access log shape: {logs.shape[0]:,} rows × {logs.shape[1]} columns")
print("\nFirst 3 URLs:")
print(logs['url'].head(3).to_string())

Access log shape: 469,977 rows × 8 columns

First 3 URLs:
0    /department/fitness/category/baseball%20&%20so...
1    /department/fan%20shop/category/hunting%20&%20...
2    /department/apparel/category/featured%20shops/...


In [8]:
# Extract department, category, and product from URL path
# Pattern matches the segment after each path keyword
logs['url_department'] = logs['url'].str.extract(
    r'/department/([^/]+)/', expand=False
)
logs['url_category'] = logs['url'].str.extract(
    r'/category/([^/]+)/', expand=False
)
logs['url_product'] = logs['url'].str.extract(
    r'/product/(.+)$', expand=False
)

# Decode percent-encoding — replace %20 with space, %26 with &, %27 with '
def decode_url(series):
    return (series
            .str.replace(r'%20', ' ', regex=False)
            .str.replace(r'%26', '&', regex=False)
            .str.replace(r'%27', "'", regex=False)
            .str.replace(r'%2B', '+', regex=False)
            .str.strip()
            .str.title())

logs['url_department'] = decode_url(logs['url_department'])
logs['url_category']   = decode_url(logs['url_category'])
logs['url_product']    = decode_url(logs['url_product'])

# Validate extraction — check for any rows where extraction failed
print("=== Extraction success rate ===")
print(f"Department extracted: {logs['url_department'].notna().sum():,} / {len(logs):,}")
print(f"Category extracted:   {logs['url_category'].notna().sum():,} / {len(logs):,}")
print(f"Product extracted:    {logs['url_product'].notna().sum():,} / {len(logs):,}")

print("\n=== Sample of parsed URLs ===")
print(logs[['url', 'url_department', 'url_category', 'url_product']].head(5).to_string())

print("\n=== All browsed departments ===")
print(logs['url_department'].value_counts().to_string())

=== Extraction success rate ===
Department extracted: 469,977 / 469,977
Category extracted:   469,977 / 469,977
Product extracted:    469,977 / 469,977

=== Sample of parsed URLs ===
                                                                                                              url url_department         url_category                                 url_product
0  /department/fitness/category/baseball%20&%20softball/product/adidas%20Brazuca%202017%20Official%20Match%20Ball        Fitness  Baseball & Softball     Adidas Brazuca 2017 Official Match Ball
1  /department/fan%20shop/category/hunting%20&%20shooting/product/The%20North%20Face%20Women's%20Recon%20Backpack       Fan Shop   Hunting & Shooting       The North Face Women'S Recon Backpack
2        /department/apparel/category/featured%20shops/product/adidas%20Kids'%20RG%20III%20Mid%20Football%20Cleat        Apparel       Featured Shops      Adidas Kids' Rg Iii Mid Football Cleat
3        /department/footwear/category/el

### Section 3 Findings

**Part A — Text Cleaning**

All categorical text columns are now standardised to consistent title case with excess whitespace removed. Key observations:

- `Customer Segment` contains three clean values: Consumer, Corporate, Home Office
- `Delivery Status` has four values: Advance Shipping, Late Delivery, Shipping Canceled, Shipping On Time
- `Shipping Mode` has four values: First Class, Same Day, Second Class, Standard Class
- `Market` has five regions — USCA required a manual acronym fix after title casing incorrectly produced "Usca"
- `Order Region` spans 23 distinct regions globally — four US-specific regions (East of USA, South of USA, West of USA, US Center) also required manual acronym correction

**Part B — URL Parsing**

The access log contains 469,977 browsing events across 6 departments. Regex extraction succeeded on 100% of rows — every URL followed the consistent `/department/.../category/.../product/...` path structure with no malformed entries.

Outdoors is the most browsed department with 79,926 visits, followed closely by Apparel and Footwear. All six departments show broadly similar traffic volumes — no single department dominates browsing behaviour.

The relationship between browse volume and actual purchase revenue will be examined in the EDA notebook.

## Section 4 — Feature Engineering

This section derives new columns from existing data that do not exist in the raw file but are analytically useful. None of these columns introduce external information — everything is calculated from what the dataset already contains.

Three features are engineered here:

- **`profit_margin_pct`** — profit as a percentage of sales per order line. The raw dataset has `Benefit per order` and `Sales per customer` separately but no margin column. Margin is the metric operations and finance teams actually track.
- **`on_time_delivery`** — a clean binary flag derived from `Delivery Status` for use as a simple filter in Power BI. More readable than the numeric `is_late` flag for dashboard labels.
- **`order_value_band`** — sales per customer binned into four quartile-based tiers. Allows Power BI to answer whether high-value customers receive better delivery performance without requiring the viewer to interpret raw sales numbers.

In [9]:
# Profit margin as percentage of sales
# Guard against division by zero on any zero-sales rows
df['profit_margin_pct'] = np.where(
    df['Sales per customer'] != 0,
    (df['Benefit per order'] / df['Sales per customer'] * 100).round(2),
    0
)

# Clean on-time binary flag from Delivery Status
# Advance Shipping and Shipping On Time = on time
# Late Delivery and Shipping Canceled = not on time
df['on_time_delivery'] = df['Delivery Status'].isin(
    ['Advance Shipping', 'Shipping On Time']
).astype(int)

# Order value band — quartile based
df['order_value_band'] = pd.qcut(
    df['Sales per customer'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Premium']
)

print("=== Profit Margin Summary ===")
print(df['profit_margin_pct'].describe().round(2))

print("\n=== On Time Delivery Rate ===")
on_time_rate = df['on_time_delivery'].mean() * 100
print(f"{on_time_rate:.1f}% of orders delivered on time or early")

print("\n=== Order Value Band Distribution ===")
print(df['order_value_band'].value_counts().sort_index().to_string())


print("\n=== On Time Rate by Order Value Band ===")
print(df.groupby('order_value_band', observed=True)['on_time_delivery']
      .mean()
      .mul(100)
      .round(1)
      .to_string())

=== Profit Margin Summary ===
count    180508.00
mean         12.04
std          46.67
min        -275.08
25%           7.50
50%          27.00
75%          36.30
max          50.06
Name: profit_margin_pct, dtype: float64

=== On Time Delivery Rate ===
40.9% of orders delivered on time or early

=== Order Value Band Distribution ===
order_value_band
Low        45305
Medium     45263
High       44867
Premium    45073

=== On Time Rate by Order Value Band ===
order_value_band
Low        40.6
Medium     40.8
High       41.1
Premium    41.0


### Order Value Band — Boundary Verification

Quartile-based binning produces four equal-sized groups by row count, not by dollar width. The table below documents the actual sales ranges behind each label so the bands can be interpreted correctly in the analysis and dashboard.

In [10]:
print("=== Order Value Band — Actual Sales Ranges ===")
print(df.groupby('order_value_band', observed=True)['Sales per customer']
      .agg(['min', 'max', 'mean'])
      .round(2)
      .to_string())

=== Order Value Band — Actual Sales Ranges ===
                     min      max    mean
order_value_band                         
Low                 7.49   104.38   65.27
Medium            104.39   163.99  129.24
High              164.00   247.40  198.81
Premium           247.49  1939.99  339.99


### Section 4 Findings

**Profit margin** averages 12% across all orders but the standard deviation of 46.7 points tells the more important story — there is enormous variability. The minimum of -275% indicates orders where discounts or returns produced significant losses, while the maximum caps at 50%. The median of 27% is more representative of typical order profitability than the mean, which is pulled down by loss-making outliers.

**On-time delivery rate** is 40.9%. This means the majority of orders — 59.1% — are either late or cancelled. This is the single most operationally significant finding in the dataset and will anchor the Logistics page of the Power BI dashboard.

**Order value bands** divide into four roughly equal groups of approximately 45,000 orders each. The dollar ranges behind each label are:

| Band | Sales Range | Mean |
|---|---|---|
| Low | $7.49 — $104.38 | $65 |
| Medium | $104.39 — $163.99 | $129 |
| High | $164.00 — $247.40 | $199 |
| Premium | $247.49 — $1,939.99 | $340 |

**On-time rate by order value band** shows virtually no difference across tiers — Low orders deliver on time 40.6% of the time, Premium orders 41.0%. The delivery performance problem is uniform across all customer spend levels. This is a meaningful finding: the late delivery issue is systemic, not concentrated in low-value orders that might be deprioritised. High-spending customers are experiencing the same failure rate as low-spending ones.

## Section 5 — Null Handling & Validation

The major null decisions were made earlier in this notebook. This section serves two purposes: confirming that no new nulls were introduced during feature engineering, and documenting every null treatment decision in one place for the data quality record.

A complete null audit is run across all remaining columns before any data is exported.

### Null Treatment Decisions

| Column | Nulls | Treatment | Reason |
|---|---|---|---|
| `Product Description` | 180,519 (100%) | Column dropped | Entirely empty — no recoverable information |
| `Order Zipcode` | 155,679 (86%) | Column dropped | Unrecoverable — too sparse to be useful |
| `Customer Password` | 0 | Column dropped | PII — no analytical value |
| `Customer Email` | 0 | Column dropped | PII — no analytical value |
| `Product Image` | 0 | Column dropped | URL strings — no analytical value |
| `Customer Lname` | 8 | Rows dropped | Negligible count — 0.004% of dataset |
| `Customer Zipcode` | 3 | Rows dropped | Negligible count — 0.002% of dataset |

In [11]:
# Check for nulls across all remaining columns
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct
})

print("=== Columns With Nulls ===")
has_nulls = null_summary[null_summary['null_count'] > 0]

if len(has_nulls) == 0:
    print("No nulls found across any column.")
else:
    print(has_nulls.to_string())

print(f"\n=== Final Dataset Shape ===")
print(f"{df.shape[0]:,} rows × {df.shape[1]} columns")

# Check engineered columns specifically
engineered_cols = [
    'actual_lead_time',
    'delivery_delay_days',
    'is_late',
    'lead_time_band',
    'order_year',
    'order_month',
    'order_quarter',
    'order_yearmonth',
    'profit_margin_pct',
    'on_time_delivery',
    'order_value_band'
]

print("\n=== Engineered Column Null Check ===")
for col in engineered_cols:
    n = df[col].isnull().sum()
    status = "OK" if n == 0 else f"WARNING — {n} nulls"
    print(f"{col:<25} {status}")

=== Columns With Nulls ===
No nulls found across any column.

=== Final Dataset Shape ===
180,508 rows × 59 columns

=== Engineered Column Null Check ===
actual_lead_time          OK
delivery_delay_days       OK
is_late                   OK
lead_time_band            OK
order_year                OK
order_month               OK
order_quarter             OK
order_yearmonth           OK
profit_margin_pct         OK
on_time_delivery          OK
order_value_band          OK


The final dataset stands at 180,508 rows and 59 columns — down from the original 53 columns after dropping 5 analytically useless columns, and up to 59 after adding 11 engineered columns across Sections 2, 3, and 4.

## Section 6 — Star Schema Split

The cleaned flat file is a single denormalized table where customer, product, and shipping information repeats on every row. Loading this directly into Power BI produces a bloated, unstructured model with no relationships between entities.

The correct approach is a star schema — one central fact table containing numeric measures and foreign keys, surrounded by dimension tables containing descriptive attributes. This is how BI tools are designed to work. Relationships defined in Power BI between these tables replace the repeated columns, reduce file size, and enable cleaner DAX measures.

The schema we are building:

- **`fact_orders`** — one row per order line. Contains all numeric measures and foreign keys only.
- **`dim_customer`** — one row per unique customer. Contains all customer descriptive attributes.
- **`dim_product`** — one row per unique product. Contains all product descriptive attributes.
- **`dim_shipping`** — one row per unique shipping mode and delivery status combination.

A date dimension (`dim_date`) will be built directly in Power BI using DAX — this is standard practice and does not need to be exported from Python.

### Defining the Tables

Before splitting, we identify exactly which columns belong to each table. Foreign keys remain in `fact_orders` as the join keys — they are not removed from the fact table.

In [12]:
# --- Dim Customer ---
dim_customer_cols = [
    'Order Customer Id',
    'Customer Fname',
    'Customer Lname',
    'Customer Segment',
    'Customer City',
    'Customer State',
    'Customer Country',
    'Customer Street',
    'Customer Zipcode'
]

dim_customer = (df[dim_customer_cols]
                .drop_duplicates(subset=['Order Customer Id'])
                .reset_index(drop=True))

print(f"dim_customer: {dim_customer.shape[0]:,} rows × {dim_customer.shape[1]} columns")

# --- Dim Product ---
dim_product_cols = [
    'Product Card Id',
    'Product Name',
    'Product Price',
    'Product Status',
    'Category Id',
    'Category Name',
    'Department Id',
    'Department Name'
]

dim_product = (df[dim_product_cols]
               .drop_duplicates(subset=['Product Card Id'])
               .reset_index(drop=True))

print(f"dim_product:  {dim_product.shape[0]:,} rows × {dim_product.shape[1]} columns")

# --- Dim Shipping ---
dim_shipping_cols = [
    'Shipping Mode',
    'Delivery Status'
]

dim_shipping = (df[dim_shipping_cols]
                .drop_duplicates()
                .reset_index(drop=True)
                .reset_index()
                .rename(columns={'index': 'Shipping Id'}))

print(f"dim_shipping: {dim_shipping.shape[0]:,} rows × {dim_shipping.shape[1]} columns")
print("\nDim Shipping contents:")
print(dim_shipping.to_string())

dim_customer: 20,641 rows × 9 columns
dim_product:  118 rows × 8 columns
dim_shipping: 12 rows × 3 columns

Dim Shipping contents:
    Shipping Id   Shipping Mode    Delivery Status
0             0  Standard Class   Advance Shipping
1             1  Standard Class      Late Delivery
2             2  Standard Class   Shipping On Time
3             3  Standard Class  Shipping Canceled
4             4     First Class      Late Delivery
5             5    Second Class      Late Delivery
6             6    Second Class  Shipping Canceled
7             7        Same Day   Shipping On Time
8             8        Same Day      Late Delivery
9             9        Same Day  Shipping Canceled
10           10    Second Class   Shipping On Time
11           11     First Class  Shipping Canceled


In [13]:
# Join Shipping Id back to main df so fact table has the foreign key
df = df.merge(
    dim_shipping[['Shipping Id', 'Shipping Mode', 'Delivery Status']],
    on=['Shipping Mode', 'Delivery Status'],
    how='left'
)

# Define fact table columns
fact_cols = [
    # Keys
    'Order Id',
    'Order Customer Id',
    'Product Card Id',
    'Shipping Id',
    # Dates
    'order date (DateOrders)',
    'shipping date (DateOrders)',
    'order_year',
    'order_month',
    'order_quarter',
    'order_yearmonth',
    # Operational measures
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'actual_lead_time',
    'delivery_delay_days',
    'lead_time_band',
    'is_late',
    'on_time_delivery',
    'Late_delivery_risk',
    # Financial measures
    'Sales per customer',
    'Benefit per order',
    'profit_margin_pct',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Item Quantity',
    'Sales',
    'Order Item Total',
    'Order Profit Per Order',
    'Order Item Profit Ratio',
    # Order attributes
    'Type',
    'Order Status',
    'Order Region',
    'Order City',
    'Order Country',
    'Order State',
    'Market',
    'order_value_band'
]

fact_orders = df[fact_cols].copy()

print(f"fact_orders: {fact_orders.shape[0]:,} rows × {fact_orders.shape[1]} columns")

# Referential integrity checks
print("\n=== Referential Integrity Checks ===")
print(f"Customer IDs in fact not in dim_customer: "
      f"{fact_orders['Order Customer Id'].isin(dim_customer['Order Customer Id']).sum() == len(fact_orders)}")
print(f"Product IDs in fact not in dim_product:   "
      f"{fact_orders['Product Card Id'].isin(dim_product['Product Card Id']).sum() == len(fact_orders)}")
print(f"Shipping IDs in fact not in dim_shipping: "
      f"{fact_orders['Shipping Id'].isin(dim_shipping['Shipping Id']).sum() == len(fact_orders)}")

fact_orders: 180,508 rows × 36 columns

=== Referential Integrity Checks ===
Customer IDs in fact not in dim_customer: True
Product IDs in fact not in dim_product:   True
Shipping IDs in fact not in dim_shipping: True


### Export All Tables

In [14]:
import os

output_path = 'Data/Cleaned/'
os.makedirs(output_path, exist_ok=True)

fact_orders.to_csv(f'{output_path}fact_orders.csv', index=False)
dim_customer.to_csv(f'{output_path}dim_customer.csv', index=False)
dim_product.to_csv(f'{output_path}dim_product.csv', index=False)
dim_shipping.to_csv(f'{output_path}dim_shipping.csv', index=False)

print("=== Export Complete ===")
print(f"fact_orders:  {fact_orders.shape[0]:,} rows × {fact_orders.shape[1]} columns")
print(f"dim_customer: {dim_customer.shape[0]:,} rows × {dim_customer.shape[1]} columns")
print(f"dim_product:  {dim_product.shape[0]:,} rows × {dim_product.shape[1]} columns")
print(f"dim_shipping: {dim_shipping.shape[0]:,} rows × {dim_shipping.shape[1]} columns")
print(f"\nFiles saved to: {output_path}")

=== Export Complete ===
fact_orders:  180,508 rows × 36 columns
dim_customer: 20,641 rows × 9 columns
dim_product:  118 rows × 8 columns
dim_shipping: 12 rows × 3 columns

Files saved to: Data/Cleaned/


### Section 6 Findings

The flat file has been successfully decomposed into a star schema with four tables:

| Table | Rows | Columns | Key |
|---|---|---|---|
| `fact_orders` | 180,508 | 36 | `Order Id` |
| `dim_customer` | 20,641 | 9 | `Order Customer Id` |
| `dim_product` | 118 | 8 | `Product Card Id` |
| `dim_shipping` | 12 | 3 | `Shipping Id` |

All three referential integrity checks passed — every foreign key in `fact_orders` has a matching row in its corresponding dimension table. There are no orphaned transactions.

The 12-row `dim_shipping` table reveals that not all shipping mode and delivery status combinations exist in the data — for example, First Class orders never appear with Advance Shipping status. This is a real operational pattern in the dataset, not a data quality issue.

#### Cleaning Complete

Four clean CSVs have been exported to `Data/Cleaned/`. The next stage of this project is the EDA notebook — `supply_chain_EDA.ipynb` — which loads these files and explores the operational and commercial patterns in the data.